# 🤖 โมเดลที่ 2: WangchanBERTa Multi-Label Classification
**ม.พะเยา | 8 หมวดหมู่ปัญหา | Multi-Label NLP Model**

## 1. นำเข้าไลบรารีและตั้งค่าระบบ

In [ ]:
import os, sys, json, torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'TH Sarabun New'
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    accuracy_score, f1_score, hamming_loss,
    classification_report, multilabel_confusion_matrix
)
import seaborn as sns

# ── ตั้งค่าเส้นทางไฟล์ ──────────────────────────────────────
BASE_DIR  = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR  = os.path.join(BASE_DIR, "data", "wangchanberta")
MODEL_DIR = os.path.join(BASE_DIR, "models", "wangchanberta-up-multilabel")
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"✅ PyTorch Version: {torch.__version__}")
print(f"✅ Device: {DEVICE.type.upper()}")
print(f"✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU Only'}")
print(f"✅ Model Path: {MODEL_DIR}")


## 2. โหลดข้อมูลและตรวจสอบการกระจายของ Label

In [ ]:
import csv

CATEGORIES = [
    "อาคารและสิ่งอำนวยความสะดวก",
    "ระบบเครือข่ายและเทคโนโลยี",
    "การเรียนการสอนและวิชาการ",
    "ภูมิทัศน์และความสะอาด",
    "ความปลอดภัยและจราจร",
    "บริการทั่วไปและสวัสดิการ",
    "การเดินทางและระบบขนส่ง",
    "สุขอนามัยและความปลอดภัยทางอาหาร"
]

# โหลด CSV
train_file = os.path.join(DATA_DIR, "train.csv")
test_file  = os.path.join(DATA_DIR, "test.csv")

def load_csv(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(row)
    return rows

train_data = load_csv(train_file)
test_data  = load_csv(test_file)
print(f"✅ Train samples: {len(train_data)}")
print(f"✅ Test  samples: {len(test_data)}")

# นับ Label Distribution
def count_labels(data, categories):
    counts = Counter()
    for row in data:
        for i, cat in enumerate(categories):
            col = f"label_{i+1}"
            if row.get(col, "0").strip() == "1":
                counts[cat] += 1
    return counts

train_counts = count_labels(train_data, CATEGORIES)
test_counts  = count_labels(test_data, CATEGORIES)

# ── Plot ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Class Distribution: WangchanBERTa Dataset (ม.พะเยา)", fontsize=14, fontweight='bold')

for ax, (counts, title) in zip(axes, [(train_counts, f"Train Set (n={len(train_data)})"), (test_counts, f"Test Set (n={len(test_data)})")]):
    names  = [c[:12] + ".." if len(c) > 14 else c for c in CATEGORIES]
    values = [counts[c] for c in CATEGORIES]
    bars = ax.bar(names, values, color=plt.cm.Set2(range(8)), edgecolor='white', linewidth=1.5)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_ylabel("Count")
    ax.tick_params(axis='x', rotation=35, labelsize=8)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5, str(v), ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=150, bbox_inches='tight')
plt.show()
print("\n📊 Train Set Distribution:")
for cat in CATEGORIES:
    pct = train_counts[cat] / len(train_data) * 100
    print(f"  • {cat:<30}: {train_counts[cat]:>3} รายการ ({pct:.1f}%)")
print(f"  >> รวมทั้งหมด: {len(train_data)} รายการ")


## 3. เตรียม Dataset และ DataLoader

In [ ]:
class UPTextDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=256):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row  = self.data[idx]
        text = row.get("text", row.get("description", ""))
        labels = [float(row.get(f"label_{i+1}", 0)) for i in range(8)]
        enc = self.tokenizer(
            text, truncation=True, padding="max_length",
            max_length=self.max_len, return_tensors="pt"
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels":         torch.tensor(labels, dtype=torch.float)
        }

MODEL_NAME = MODEL_DIR if os.path.exists(MODEL_DIR) else "airesearch/wangchanberta-base-att-spm-uncased"
print(f"📦 Loading tokenizer from: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = UPTextDataset(train_data, tokenizer)
test_ds  = UPTextDataset(test_data,  tokenizer)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False)

print(f"✅ Train batches: {len(train_loader)}")
print(f"✅ Test  batches: {len(test_loader)}")


## 4. โหลดโมเดลและตั้งค่า Optimizer (Fine-Tuning)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=8, problem_type="multi_label_classification"
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
criterion = torch.nn.BCEWithLogitsLoss()
EPOCHS = 6

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model: WangchanBERTa Multi-Label (8 Classes)")
print(f"✅ Trainable Parameters: {total_params:,}")
print(f"✅ Loss Function: BCEWithLogitsLoss")
print(f"✅ Optimizer: AdamW (lr=3e-5)")
print(f"✅ Epochs: {EPOCHS}")


## 5. Training Loop — เทรนโมเดล (6 Epochs)

In [ ]:
import time

THRESHOLD = 0.50
history = {"train_loss": [], "val_loss": [], "val_f1": [], "val_acc": []}
best_val_f1 = 0.0

print("=" * 68)
print("🚀 START TRAINING: WangchanBERTa Multi-Label (UP Connect)")
print("=" * 68)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    # ── Train ──────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = criterion(outputs.logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # ── Validation ─────────────────────────────────────────────
    model.eval()
    val_loss, all_preds, all_labels = 0.0, [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss    = criterion(outputs.logits, labels)
            val_loss += loss.item()
            preds = (torch.sigmoid(outputs.logits) >= THRESHOLD).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    val_loss /= len(test_loader)
    val_f1   = f1_score(all_labels, all_preds, average="micro", zero_division=0)
    val_acc  = accuracy_score(
        np.array(all_labels).flatten().astype(int),
        np.array(all_preds).flatten().astype(int)
    )
    elapsed = time.time() - t0

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(val_f1)
    history["val_acc"].append(val_acc)

    marker = " ← 🏆 Best!" if val_f1 > best_val_f1 else ""
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), "best_wangchanberta.pth")
    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val F1: {val_f1:.4f} | "
          f"Val Acc: {val_acc:.4f} | "
          f"{elapsed:.1f}s{marker}")

print("=" * 68)
print(f"✅ Training Complete! Best Val F1: {best_val_f1:.4f}")
print("=" * 68)


## 6. กราฟ Training Loss & F1-Score Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("WangchanBERTa Training History (ม.พะเยา)", fontsize=14, fontweight='bold')

# Loss Curve
axes[0].plot(history["train_loss"], 'o-', color='#e74c3c', label='Train Loss', linewidth=2)
axes[0].plot(history["val_loss"],   's--', color='#3498db', label='Val Loss',   linewidth=2)
axes[0].set_title("BCEWithLogitsLoss per Epoch", fontweight='bold')
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend(); axes[0].grid(alpha=0.3)
for i, (tl, vl) in enumerate(zip(history["train_loss"], history["val_loss"])):
    axes[0].annotate(f"{tl:.3f}", (i, tl), textcoords="offset points", xytext=(0,8), fontsize=8, color='#e74c3c')

# F1-Score Curve  
axes[1].plot(history["val_f1"],  'o-', color='#2ecc71', label='Val Micro-F1', linewidth=2)
axes[1].plot(history["val_acc"], 's--', color='#9b59b6', label='Val Accuracy', linewidth=2)
axes[1].set_title("Micro-F1 & Accuracy per Epoch", fontweight='bold')
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Score")
axes[1].set_ylim(0, 1); axes[1].legend(); axes[1].grid(alpha=0.3)
for i, f1 in enumerate(history["val_f1"]):
    axes[1].annotate(f"{f1:.3f}", (i, f1), textcoords="offset points", xytext=(0,8), fontsize=8, color='#2ecc71')

plt.tight_layout()
plt.savefig("wangchanberta_training_curve.png", dpi=150, bbox_inches='tight')
plt.show()


## 7. สรุปผลการประเมิน — Classification Report & Confusion Matrix

In [ ]:
# ── Reload best model ──────────────────────────────────────────
model.load_state_dict(torch.load("best_wangchanberta.pth", map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds   = (torch.sigmoid(outputs.logits) >= THRESHOLD).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(batch["labels"].numpy())

y_true = np.array(all_labels)
y_pred = np.array(all_preds)

# ── Metrics Summary ────────────────────────────────────────────
micro_f1   = f1_score(y_true, y_pred, average="micro",  zero_division=0)
macro_f1   = f1_score(y_true, y_pred, average="macro",  zero_division=0)
h_loss     = hamming_loss(y_true, y_pred)

print("=" * 68)
print("📊 WANGCHANBERTA EVALUATION RESULTS (Test Set)")
print("=" * 68)
print(f"  Micro-averaged F1-Score : {micro_f1:.4f} ({micro_f1*100:.2f}%)")
print(f"  Macro-averaged F1-Score : {macro_f1:.4f} ({macro_f1*100:.2f}%)")
print(f"  Hamming Loss            : {h_loss:.4f} (ผิดพลาดเพียง {h_loss*100:.2f}%)")
print()

SHORT_CATS = [
    "อาคาร/สิ่งอำนวย", "เครือข่าย/IT", "วิชาการ",
    "ภูมิทัศน์", "ความปลอดภัย", "บริการทั่วไป",
    "ขนส่ง", "สุขอนามัย"
]
report = classification_report(y_true, y_pred, target_names=SHORT_CATS, zero_division=0, output_dict=True)
df_report = pd.DataFrame(report).T.drop(["accuracy", "macro avg", "weighted avg"], errors='ignore')
df_report = df_report[["precision", "recall", "f1-score", "support"]].round(4)
print(df_report.to_string())
print()

# ── Confusion Matrix (Per Class) ──────────────────────────────
cms = multilabel_confusion_matrix(y_true, y_pred)
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle("Confusion Matrix — WangchanBERTa (ทั้ง 8 หมวดหมู่)", fontsize=14, fontweight='bold')
for i, (ax, cm, name) in enumerate(zip(axes.flat, cms, SHORT_CATS)):
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues',
                xticklabels=["Pred 0", "Pred 1"], yticklabels=["True 0", "True 1"])
    ax.set_title(f"{i+1}. {name}", fontsize=10, fontweight='bold')
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig("wangchanberta_confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Per-Class F1 Bar Chart ─────────────────────────────────────
per_class_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#2ecc71' if f >= 0.90 else '#f39c12' if f >= 0.80 else '#e74c3c' for f in per_class_f1]
bars = ax.bar(SHORT_CATS, per_class_f1, color=colors, edgecolor='white', linewidth=1.5)
ax.axhline(y=micro_f1, color='#3498db', linestyle='--', linewidth=2, label=f'Micro-F1 = {micro_f1:.4f}')
ax.set_ylim(0, 1.1); ax.set_ylabel("F1-Score"); ax.set_title("Per-Class F1-Score", fontweight='bold')
ax.tick_params(axis='x', rotation=30)
ax.legend()
for bar, val in zip(bars, per_class_f1):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01, f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig("wangchanberta_f1_per_class.png", dpi=150, bbox_inches='tight')
plt.show()
